In [13]:
import polars as pl
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
# !pip install lets-plot
# from lets_plot import *
# LetsPlot.setup_html()

bikes = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv")
# from lets_plot import *
# LetsPlot.setup_html()

In [14]:
# ── 2. Create target variable ─────────────────────────────────────────────────
bikes = bikes.with_columns(
    (pl.col("casual") + pl.col("registered")).alias("cnt")
)

# ── 3. Extract year and month from date ───────────────────────────────────────
bikes = bikes.with_columns([
    pl.col("dteday").str.to_date(format="%m/%d/%Y").dt.year().alias("yr"),
    pl.col("dteday").str.to_date(format="%m/%d/%Y").dt.month().alias("mnth"),
])

# ── 4. Add COVID flag ─────────────────────────────────────────────────────────
bikes = bikes.with_columns(
    pl.col("yr").is_in([2020, 2021]).cast(pl.Int8).alias("is_covid")
)

# ── 5. Add Black Friday flag and fix workingday ───────────────────────────────
from datetime import date
import calendar

def get_black_friday(year):
    c = calendar.monthcalendar(year, 11)
    thursdays = [week[3] for week in c if week[3] != 0]
    thanksgiving_day = thursdays[3]
    black_friday_day = thanksgiving_day + 1
    return date(year, 11, black_friday_day).strftime("%m/%d/%Y")

black_fridays = [get_black_friday(y) for y in range(2011, 2024)]

bikes = bikes.with_columns(
    pl.col("dteday").is_in(black_fridays).cast(pl.Int8).alias("is_black_friday")
)

# Fix workingday for Black Friday — it's not a real working day
bikes = bikes.with_columns(
    pl.when(pl.col("is_black_friday") == 1)
    .then(0)
    .otherwise(pl.col("workingday"))
    .alias("workingday")
)

# ── 6. Cyclical encoding ──────────────────────────────────────────────────────
bikes = bikes.with_columns([
    (2 * np.pi * pl.col("hr") / 24).sin().alias("hr_sin"),
    (2 * np.pi * pl.col("hr") / 24).cos().alias("hr_cos"),
    (2 * np.pi * (pl.col("mnth") - 1) / 12).sin().alias("mnth_sin"),
    (2 * np.pi * (pl.col("mnth") - 1) / 12).cos().alias("mnth_cos"),
])

# ── 6b. Interaction features ──────────────────────────────────────────────────
bikes = bikes.with_columns([
    (pl.col("hr_sin") * pl.col("feels_like_c")).alias("hr_sin_x_temp"),
    (pl.col("hr_cos") * pl.col("feels_like_c")).alias("hr_cos_x_temp"),
])

# ── 7. One-hot encode season ──────────────────────────────────────────────────
bikes = bikes.to_dummies(columns=["season"])

# ── 8. Drop columns no longer needed ─────────────────────────────────────────
bikes = bikes.drop([
    "dteday", "hr", "mnth",
    "temp_c",
    "casual", "registered",
])

# ── 9. Train/test split ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

feature_cols = [col for col in bikes.columns if col != "cnt"]

X = bikes.select(feature_cols).to_numpy()
y = bikes.select("cnt").to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 10. Scale numeric features ─────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler

cols_to_scale = ["feels_like_c", "windspeed", "hum"]
scale_idx = [feature_cols.index(c) for c in cols_to_scale]

feature_scaler = MinMaxScaler()
X_train[:, scale_idx] = feature_scaler.fit_transform(X_train[:, scale_idx])
X_test[:, scale_idx]  = feature_scaler.transform(X_test[:, scale_idx])

In [ ]:
# # ── 2. Create target variable ─────────────────────────────────────────────────
# bikes = bikes.with_columns(
#     (pl.col("casual") + pl.col("registered")).alias("cnt")
# )

# # ── 3. Extract year and month from date ───────────────────────────────────────
# bikes = bikes.with_columns([
#     pl.col("dteday").str.to_date(format="%m/%d/%Y").dt.year().alias("yr"),
#     pl.col("dteday").str.to_date(format="%m/%d/%Y").dt.month().alias("mnth"),
# ])

# # ── 4. Add COVID flag ─────────────────────────────────────────────────────────
# bikes = bikes.with_columns(
#     pl.col("yr").is_in([2020, 2021]).cast(pl.Int8).alias("is_covid")
# )

# # ── 5. Cyclical encoding ──────────────────────────────────────────────────────
# bikes = bikes.with_columns([
#     (2 * np.pi * pl.col("hr") / 24).sin().alias("hr_sin"),
#     (2 * np.pi * pl.col("hr") / 24).cos().alias("hr_cos"),
#     (2 * np.pi * (pl.col("mnth") - 1) / 12).sin().alias("mnth_sin"),
#     (2 * np.pi * (pl.col("mnth") - 1) / 12).cos().alias("mnth_cos"),
# ])

# # ── 6. One-hot encode season ──────────────────────────────────────────────────
# bikes = bikes.to_dummies(columns=["season"])

# # ── 7. Drop columns no longer needed ─────────────────────────────────────────
# bikes = bikes.drop([
#     "dteday", "hr", "mnth",
#     "temp_c",
#     "casual", "registered",
# ])

# # ── 8. Train/test split ───────────────────────────────────────────────────────
# from sklearn.model_selection import train_test_split

# feature_cols = [col for col in bikes.columns if col != "cnt"]

# X = bikes.select(feature_cols).to_numpy()
# y = bikes.select("cnt").to_numpy()

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # ── 9. Scale numeric features ─────────────────────────────────────────────────
# from sklearn.preprocessing import MinMaxScaler

# cols_to_scale = ["feels_like_c", "windspeed", "hum"]
# scale_idx = [feature_cols.index(c) for c in cols_to_scale]

# feature_scaler = MinMaxScaler()
# X_train[:, scale_idx] = feature_scaler.fit_transform(X_train[:, scale_idx])
# X_test[:, scale_idx]  = feature_scaler.transform(X_test[:, scale_idx])

In [15]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import random

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# ── 1. Build the model ────────────────────────────────────────────────────────
model = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1)
])

# ── 2. Compile the model ──────────────────────────────────────────────────────
model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mae']
)

# ── 3. Train the model ────────────────────────────────────────────────────────
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=500,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/500


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2812/2812 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 66554.6094 - mae: 183.4089 - val_loss: 51958.2227 - val_mae: 152.5232
Epoch 2/500
2812/2812 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 50436.2891 - mae: 153.4028 - val_loss: 48349.9102 - val_mae: 145.2090
Epoch 3/500
2812/2812 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 46232.5977 - mae: 145.3841 - val_loss: 43997.8633 - val_mae: 138.6377
Epoch 4/500
2812/2812 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 41774.3516 - mae: 137.8380 - val_loss: 41326.5156 - val_mae: 133.4432
Epoch 5/500
2812/2812 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 35677.1328 - mae: 126.9842 - val_loss: 33916.6641 - val_mae: 121.6954
Epoch 6/500
2812/2812 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 31684.0195 - mae: 119.0237 - val_loss: 29295.5586 - val_mae: 113.8628
Epoch 7/500
2812/2812 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 30374.1719 - mae: 115.9723 - val_loss: 29256.9199 - val_mae: 113.4178
Epoch 8/500
2812/2812 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 29530.3164 - mae

In [16]:
# ── Evaluate the model ────────────────────────────────────────────────────────
loss, mae = model.evaluate(X_test, y_test)
print(f"Test MAE: {mae:.1f} riders")

703/703 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 22671.3711 - mae: 96.6281
Test MAE: 96.6 riders


In [17]:
# ── Load mini holdout ─────────────────────────────────────────────────────────
mini = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/biking_holdout_test_mini.csv")

# ── Apply the same transformations as training data ───────────────────────────
# Extract year and month
mini = mini.with_columns([
    pl.col("dteday").str.to_date(format="%m/%d/%Y").dt.year().alias("yr"),
    pl.col("dteday").str.to_date(format="%m/%d/%Y").dt.month().alias("mnth"),
])

# Add COVID flag — November 2023 is not a COVID year
mini = mini.with_columns(
    pl.lit(0).cast(pl.Int8).alias("is_covid")
)

# Add Black Friday flag
mini = mini.with_columns(
    pl.col("dteday").is_in(black_fridays).cast(pl.Int8).alias("is_black_friday")
)

# Fix workingday for Black Friday
mini = mini.with_columns(
    pl.when(pl.col("is_black_friday") == 1)
    .then(0)
    .otherwise(pl.col("workingday"))
    .alias("workingday")
)

# Cyclical encoding
mini = mini.with_columns([
    (2 * np.pi * pl.col("hr") / 24).sin().alias("hr_sin"),
    (2 * np.pi * pl.col("hr") / 24).cos().alias("hr_cos"),
    (2 * np.pi * (pl.col("mnth") - 1) / 12).sin().alias("mnth_sin"),
    (2 * np.pi * (pl.col("mnth") - 1) / 12).cos().alias("mnth_cos"),
])

# # Interaction features
mini = mini.with_columns([
    (pl.col("hr_sin") * pl.col("feels_like_c")).alias("hr_sin_x_temp"),
    (pl.col("hr_cos") * pl.col("feels_like_c")).alias("hr_cos_x_temp"),
])

# One-hot encode season
mini = mini.to_dummies(columns=["season"])

# Drop same columns as training
mini = mini.drop(["dteday", "hr", "mnth", "temp_c"])

# ── Make sure columns match training data exactly ─────────────────────────────
for col in feature_cols:
    if col not in mini.columns:
        mini = mini.with_columns(pl.lit(0).alias(col))

mini_X = mini.select(feature_cols).to_numpy()

# ── Scale numeric features using the SAME scaler from training ────────────────
mini_X[:, scale_idx] = feature_scaler.transform(mini_X[:, scale_idx])

# ── Predict ───────────────────────────────────────────────────────────────────
mini_preds = model.predict(mini_X)
mini_preds = mini_preds.clip(min=0)

# ── Save predictions ──────────────────────────────────────────────────────────
my_predictions = pl.DataFrame({
    "predictions": mini_preds.flatten()
})

my_predictions.write_csv("ctrl_alt_elite-module4-predictions.csv")

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


In [ ]:
# (
#     ggplot(bikes, aes(y='cnt', x='yr'))
#     + geom_histogram(binwidth=10)
# )

In [ ]:
# Load the answers
answers = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/biking_holdout_test_mini_answers.csv")
answers = answers.with_columns(
    (pl.col("casual") + pl.col("registered")).alias("actual")
).select("actual")

# Load your predictions
preds = pl.read_csv("ctrl_alt_elite-module4-predictions.csv")

# Load the original mini holdout to get dates
mini_raw = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/biking_holdout_test_mini.csv")

# Build comparison dataframe
comparison = mini_raw.select(["dteday", "hr", "workingday"]).with_columns([
    answers["actual"],
    preds["predictions"],
])

comparison = comparison.with_columns([
    (pl.col("actual") - pl.col("predictions")).alias("error"),
    (((pl.col("actual") - pl.col("predictions")).abs()) / pl.col("actual")).alias("pct_error")
])

# Show the worst predictions
print(comparison.sort("pct_error", descending=True).head(20))

shape: (20, 7)
┌────────────┬──────┬────────────┬────────┬─────────────┬────────────┬───────────┐
│ dteday     ┆ hr   ┆ workingday ┆ actual ┆ predictions ┆ error      ┆ pct_error │
│ ---        ┆ ---  ┆ ---        ┆ ---    ┆ ---         ┆ ---        ┆ ---       │
│ str        ┆ f64  ┆ i64        ┆ i64    ┆ f64         ┆ f64        ┆ f64       │
╞════════════╪══════╪════════════╪════════╪═════════════╪════════════╪═══════════╡
│ 11/24/2023 ┆ 6.0  ┆ 1          ┆ 49     ┆ 253.97812   ┆ -204.97812 ┆ 4.183227  │
│ 11/24/2023 ┆ 7.0  ┆ 1          ┆ 128    ┆ 597.8746    ┆ -469.8746  ┆ 3.670895  │
│ 11/29/2023 ┆ 3.0  ┆ 1          ┆ 5      ┆ 22.80975    ┆ -17.80975  ┆ 3.56195   │
│ 11/24/2023 ┆ 5.0  ┆ 1          ┆ 24     ┆ 105.05992   ┆ -81.05992  ┆ 3.377497  │
│ 11/24/2023 ┆ 8.0  ┆ 1          ┆ 220    ┆ 948.556     ┆ -728.556   ┆ 3.311618  │
│ …          ┆ …    ┆ …          ┆ …      ┆ …           ┆ …          ┆ …         │
│ 11/21/2023 ┆ 21.0 ┆ 1          ┆ 53     ┆ 129.01297   ┆ -76.01297  ┆ 1

In [ ]:
# Average error by date
daily_error = comparison.group_by("dteday").agg([
    pl.col("actual").mean().alias("avg_actual"),
    pl.col("predictions").mean().alias("avg_predicted"),
    pl.col("error").mean().alias("avg_error")
]).sort("avg_error")

print(daily_error)

shape: (16, 4)
┌────────────┬────────────┬───────────────┬────────────┐
│ dteday     ┆ avg_actual ┆ avg_predicted ┆ avg_error  │
│ ---        ┆ ---        ┆ ---           ┆ ---        │
│ str        ┆ f64        ┆ f64           ┆ f64        │
╞════════════╪════════════╪═══════════════╪════════════╡
│ 11/21/2023 ┆ 153.583333 ┆ 196.910623    ┆ -43.32729  │
│ 11/24/2023 ┆ 317.541667 ┆ 333.062       ┆ -15.520333 │
│ 11/25/2023 ┆ 310.791667 ┆ 272.409286    ┆ 38.382381  │
│ 11/23/2023 ┆ 255.625    ┆ 215.425591    ┆ 40.199409  │
│ 11/26/2023 ┆ 220.083333 ┆ 159.521701    ┆ 60.561633  │
│ …          ┆ …          ┆ …             ┆ …          │
│ 11/30/2023 ┆ 523.125    ┆ 332.364784    ┆ 190.760216 │
│ 11/16/2023 ┆ 630.125    ┆ 381.331168    ┆ 248.793832 │
│ 11/15/2023 ┆ 603.5      ┆ 343.952992    ┆ 259.547008 │
│ 11/17/2023 ┆ 617.5      ┆ 357.036528    ┆ 260.463472 │
│ 11/18/2023 ┆ 580.083333 ┆ 301.937142    ┆ 278.146191 │
└────────────┴────────────┴───────────────┴────────────┘


In [19]:
# ── Load final holdout ────────────────────────────────────────────────────────
holdout = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes_december.csv")

# ── Apply the same transformations as training data ───────────────────────────
# Extract year and month
holdout = holdout.with_columns([
    pl.col("dteday").str.to_date(format="%m/%d/%Y").dt.year().alias("yr"),
    pl.col("dteday").str.to_date(format="%m/%d/%Y").dt.month().alias("mnth"),
])

# Add COVID flag
holdout = holdout.with_columns(
    pl.lit(0).cast(pl.Int8).alias("is_covid")
)

# Add Black Friday flag
holdout = holdout.with_columns(
    pl.col("dteday").is_in(black_fridays).cast(pl.Int8).alias("is_black_friday")
)

# Fix workingday for Black Friday
holdout = holdout.with_columns(
    pl.when(pl.col("is_black_friday") == 1)
    .then(0)
    .otherwise(pl.col("workingday"))
    .alias("workingday")
)

# Cyclical encoding
holdout = holdout.with_columns([
    (2 * np.pi * pl.col("hr") / 24).sin().alias("hr_sin"),
    (2 * np.pi * pl.col("hr") / 24).cos().alias("hr_cos"),
    (2 * np.pi * (pl.col("mnth") - 1) / 12).sin().alias("mnth_sin"),
    (2 * np.pi * (pl.col("mnth") - 1) / 12).cos().alias("mnth_cos"),
])

# Interaction features
holdout = holdout.with_columns([
    (pl.col("hr_sin") * pl.col("feels_like_c")).alias("hr_sin_x_temp"),
    (pl.col("hr_cos") * pl.col("feels_like_c")).alias("hr_cos_x_temp"),
])

# One-hot encode season
holdout = holdout.to_dummies(columns=["season"])

# Drop same columns as training
holdout = holdout.drop(["dteday", "hr", "mnth", "temp_c"])

# ── Make sure columns match training data exactly ─────────────────────────────
for col in feature_cols:
    if col not in holdout.columns:
        holdout = holdout.with_columns(pl.lit(0).alias(col))

holdout_X = holdout.select(feature_cols).to_numpy()

# ── Scale using the SAME scaler from training ─────────────────────────────────
holdout_X[:, scale_idx] = feature_scaler.transform(holdout_X[:, scale_idx])

# ── Predict ───────────────────────────────────────────────────────────────────
holdout_preds = model.predict(holdout_X)
holdout_preds = holdout_preds.clip(min=0)

# ── Save predictions ──────────────────────────────────────────────────────────
my_predictions = pl.DataFrame({
    "predictions": holdout_preds.flatten()
})

my_predictions.write_csv("ctrl_alt_elite-module4-predictions.csv")
print(f"Predictions saved! Total rows: {len(my_predictions)}")

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Predictions saved! Total rows: 1465
